# 01 — Data Cleaning & Preparation
**Dataset:** Our World in Data — Energy Dataset (Synthetic, based on real-world patterns)
**Purpose:** Loading the raw energy dataset, cleaning the data, extracting new features.

---
This notebook covers:
1. Loading and inspecting the raw CSV
2. Filtering rows (regions, years, data completeness)
3. Handling missing values
4. Feature engineering (derived columns)
5. Saving the cleaned dataset

## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_RAW  = os.path.join('..', 'data', 'owid-energy-data.csv')
DATA_CLEAN = os.path.join('..', 'data', 'owid-energy-cleaned.csv')

print('Libraries loaded successfully.')

AttributeError: partially initialized module 'pandas' has no attribute '_pandas_parser_CAPI' (most likely due to a circular import)

## 2. Load Raw Data

In [ ]:
df_raw = pd.read_csv(DATA_RAW)

print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'\nColumn names:')
for col in df_raw.columns:
    print(f'  {col}: {df_raw[col].dtype}')

## 3. Initial Inspection

In [ ]:
print('=== First 5 rows ===')
display(df_raw.head())

print('\n=== Missing values per column ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
display(missing_df[missing_df['Missing Count'] > 0])

print(f'\nUnique countries: {df_raw["country"].nunique()}')
print(f'Year range: {df_raw["year"].min()} – {df_raw["year"].max()}')

## 4. Row Filtering
- **Remove aggregated regions** (iso_code starting with 'OWID_'), keeping only real countries and the 'OWID_WRL' world total.
- **Restrict years** to 1990–2022 (reliable data range).
- **Minimum data completeness**: keep only countries with ≥20 non-null values in `renewables_share_energy`.

In [ ]:
df = df_raw.copy()
print(f'Starting shape: {df.shape}')

# Step 1: Remove OWID aggregates but keep OWID_WRL (world total)
mask_owid = df['iso_code'].str.startswith('OWID_', na=False)
mask_world = df['iso_code'] == 'OWID_WRL'
df = df[~mask_owid | mask_world]
print(f'After removing OWID aggregates: {df.shape}')

# Step 2: Filter to 1990–2022
df = df[(df['year'] >= 1990) & (df['year'] <= 2022)]
print(f'After year filter (1990-2022): {df.shape}')

# Step 3: Keep countries with at least 20 non-null renewables values
valid_countries = df.groupby('country')['renewables_share_energy'].count()
valid_countries = valid_countries[valid_countries >= 20].index
df = df[df['country'].isin(valid_countries)]
print(f'After completeness filter: {df.shape}')
print(f'Countries retained: {df["country"].nunique()}')

## 5. Missing Value Handling
- **Forward-fill** (propagate last known value forward) — handles gaps inside series.
- **Backward-fill** — handles gaps at the start of a country's series.
- **Zero-fill only** for electricity generation columns (not ratios/shares), because a missing generation figure typically means zero output rather than unknown data.

In [ ]:
nan_before = df.isnull().sum().sum()

# Forward fill + backward fill per country group (sorted by year)
df = df.sort_values(['country', 'year'])

share_cols = [c for c in df.columns if 'share' in c or 'per_capita' in c or 'per_gdp' in c]
gen_cols   = [c for c in df.columns if 'electricity' in c or 'generation' in c]

# Fill share/ratio columns with ffill + bfill per country
df[share_cols] = df.groupby('country')[share_cols].transform(lambda x: x.ffill().bfill())

# Fill generation columns with ffill + bfill, then 0 for remaining
df[gen_cols] = df.groupby('country')[gen_cols].transform(lambda x: x.ffill().bfill())
df[gen_cols] = df[gen_cols].fillna(0)

nan_after = df.isnull().sum().sum()
print(f'NaN values before filling: {nan_before:,}')
print(f'NaN values after filling:  {nan_after:,}')
print(f'Values filled: {nan_before - nan_after:,}')

remaining = df.isnull().sum()
print(f'\nColumns with remaining NaN:')
print(remaining[remaining > 0])

## 6. Feature Engineering
- `clean_share_energy`: combined solar + wind + hydro share (clean but not nuclear).
- `decade`: categorical decade label for temporal grouping.
- Ensure `continent` column is present.

In [ ]:
# Fossil share (sum of coal + oil + gas)
if 'fossil_share_energy' not in df.columns:
    df['fossil_share_energy'] = (
        df['coal_share_energy'].fillna(0) +
        df['oil_share_energy'].fillna(0) +
        df['gas_share_energy'].fillna(0)
    )

# Clean energy share (solar + wind + hydro)
df['clean_share_energy'] = (
    df['solar_share_energy'].fillna(0) +
    df['wind_share_energy'].fillna(0) +
    df['hydro_share_energy'].fillna(0)
)

# GDP per capita (if not present)
if 'gdp_per_capita' not in df.columns:
    df['gdp_per_capita'] = df['gdp'] / df['population'].replace(0, np.nan)

# Decade label
def decade_label(year):
    if year < 2000: return '1990s'
    elif year < 2010: return '2000s'
    elif year < 2020: return '2010s'
    else: return '2020s'

df['decade'] = df['year'].apply(decade_label)

print('New columns added:')
print('  fossil_share_energy, clean_share_energy, gdp_per_capita, decade')
print(f'\nFinal shape: {df.shape}')
print('\nSample of engineered columns:')
display(df[['country','year','renewables_share_energy','fossil_share_energy','clean_share_energy','decade']].head(10))

## 7. Final Summary

In [ ]:
key_cols = ['renewables_share_energy','fossil_share_energy','solar_share_energy',
            'wind_share_energy','hydro_share_energy','coal_share_energy',
            'gdp_per_capita','co2_per_capita','energy_per_gdp','electricity_generation']

print('=== FINAL DATASET SUMMARY ===')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Countries: {df["country"].nunique()}')
print(f'Years: {df["year"].min()} – {df["year"].max()}')
print(f'Continents: {df["continent"].unique()}')
print(f'\nDescriptive Statistics (key columns):')
display(df[key_cols].describe().round(2))

# Save
df.to_csv(DATA_CLEAN, index=False)
print(f'\n✅ Cleaned dataset saved to: {DATA_CLEAN}')

## 8. Quick Validation Plot

In [ ]:
world = df[df['country'] == 'World'].sort_values('year')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(world['year'], world['renewables_share_energy'], color='#27ae60', linewidth=2.5, marker='o', markersize=3)
ax.fill_between(world['year'], world['renewables_share_energy'], alpha=0.15, color='#27ae60')
ax.set_title('Global Average Renewable Energy Share (1990–2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Renewables Share (%)')
ax.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()
print('✅ Validation plot looks correct — upward trend confirmed.')